# 🏭 Project 10.2 — Production Line Interval Scheduler
**DSA for Mechatronics · Week 10 — Sorting Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from copy import deepcopy
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A production line records **maintenance windows** as time intervals [start, end].
We need to:
1. Sort intervals by start time using **merge sort** (implemented from scratch with merge trace)
2. **Merge overlapping intervals** — adjacent or overlapping windows should combine into one
3. **Merge two sorted arrays in-place** from different shifts (no extra allocation)
4. Find the **total covered time** and **largest gap** between maintenance windows


## Step 1 — Generate maintenance intervals and implement merge sort

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_INTERVALS  = random.randint(10, 16)
N_SHIFT_A    = random.randint(5, 9)
N_SHIFT_B    = random.randint(4, 7)
SHIFT_LENGTH = 480   # 8-hour shift in minutes

# Random intervals [start, end] within the shift
raw_intervals = []
for _ in range(N_INTERVALS):
    s = random.randint(0, SHIFT_LENGTH - 20)
    e = s + random.randint(10, min(60, SHIFT_LENGTH - s))
    raw_intervals.append([s, e])

print(f"Production line parameters:")
print(f"  Maintenance windows : {N_INTERVALS}")
print(f"  Shift length        : {SHIFT_LENGTH} min")
print()
print(f"  Raw intervals (unsorted):")
for i, (s, e) in enumerate(raw_intervals):
    print(f"    [{i:2d}] start={s:4d} min  end={e:4d} min  duration={e-s:3d} min")

# ── Merge sort on intervals (sort by start time) ─────────────────
merge_steps = []   # record merge operations for trace

def merge_intervals_ms(left, right):
    """Merge two sorted lists of intervals by start time."""
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i][0] <= right[j][0]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:]); result.extend(right[j:])
    merge_steps.append(f"merge {[x[0] for x in left]} ++ {[x[0] for x in right]}")
    return result

def merge_sort_intervals(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    return merge_intervals_ms(merge_sort_intervals(arr[:mid]),
                               merge_sort_intervals(arr[mid:]))

sorted_intervals = merge_sort_intervals([list(x) for x in raw_intervals])

print()
print(f"Merge sort trace (start-values merged at each step):")
for step_i, step in enumerate(merge_steps, 1):
    print(f"  step {step_i:2d}: {step}")

print()
print(f"After merge sort (by start time):")
for s, e in sorted_intervals:
    print(f"  [{s:4d}, {e:4d}]  duration={e-s} min")


## Step 2 — Merge overlapping intervals

In [ ]:
def merge_overlapping(intervals):
    """
    Merge all overlapping/touching intervals.

    Algorithm:
      1. Sort by start time (already done).
      2. Start with first interval as current.
      3. For each next interval:
         - If it overlaps or touches current (next.start <= current.end):
           extend current.end = max(current.end, next.end).
         - Else: save current, start new current.
      4. Save final current.

    O(n) after sorting. O(n) output.
    """
    if not intervals: return []
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:   # overlaps or touches
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

merged = merge_overlapping(sorted_intervals)

total_covered   = sum(e - s for s, e in merged)
gaps            = [[merged[i][1], merged[i+1][0]] for i in range(len(merged)-1)
                   if merged[i+1][0] > merged[i][1]]
largest_gap     = max((e - s for s, e in gaps), default=0)
gap_positions   = gaps

print(f"Overlapping interval merge:")
print(f"  Before: {N_INTERVALS} intervals")
print(f"  After : {len(merged)} merged windows")
print()
print(f"  {'Merged window':<20}  {'Duration':>10}")
print(f"  {'─'*20}  {'─'*10}")
for s, e in merged:
    print(f"  [{s:4d} – {e:4d}]           {e-s:10d} min")

print()
print(f"  Total covered time : {total_covered} / {SHIFT_LENGTH} min  ({total_covered/SHIFT_LENGTH*100:.1f}%)")
print(f"  Number of gaps     : {len(gaps)}")
print(f"  Largest gap        : {largest_gap} min")
if gaps:
    g_s, g_e = max(gaps, key=lambda g: g[1]-g[0])
    print(f"  Largest gap range  : [{g_s} – {g_e}]")


## Step 3 — Merge two sorted shift arrays in-place

In [ ]:
def merge_sorted_inplace(nums1, m, nums2, n):
    """
    Merge nums2 (length n) into nums1 (which has m real values + n zeros at end).

    Three-pointer approach from the back:
      p1 = m-1  (last real value in nums1)
      p2 = n-1  (last value in nums2)
      p  = m+n-1  (write position)

    Compare from the back, write larger value at p and move pointers left.
    O(m + n) time, O(1) extra space — no allocation needed.
    """
    p1, p2, p = m - 1, n - 1, m + n - 1
    steps = []
    while p2 >= 0:
        if p1 >= 0 and nums1[p1] >= nums2[p2]:
            steps.append(f"place nums1[{p1}]={nums1[p1]} at pos {p}")
            nums1[p] = nums1[p1]; p1 -= 1
        else:
            steps.append(f"place nums2[{p2}]={nums2[p2]} at pos {p}")
            nums1[p] = nums2[p2]; p2 -= 1
        p -= 1
    return nums1

# Build two sorted shift start-time arrays
shift_a = sorted(random.randint(0, SHIFT_LENGTH) for _ in range(N_SHIFT_A))
shift_b = sorted(random.randint(0, SHIFT_LENGTH) for _ in range(N_SHIFT_B))

print(f"Merge two sorted shift arrays in-place:")
print(f"  Shift A ({N_SHIFT_A} events): {shift_a}")
print(f"  Shift B ({N_SHIFT_B} events): {shift_b}")
print()

# Prepare nums1 with trailing zeros
nums1 = shift_a + [0] * N_SHIFT_B
nums2 = shift_b[:]

print(f"  nums1 before merge: {nums1}")

result = merge_sorted_inplace(nums1, N_SHIFT_A, nums2, N_SHIFT_B)

print()
print(f"  Step trace (first 8 steps):")
steps_demo = []
nums1_demo = shift_a + [0] * N_SHIFT_B
nums2_demo = shift_b[:]
p1, p2, p = N_SHIFT_A-1, N_SHIFT_B-1, N_SHIFT_A+N_SHIFT_B-1
step_count = 0
while p2 >= 0 and step_count < 8:
    if p1 >= 0 and nums1_demo[p1] >= nums2_demo[p2]:
        print(f"    place nums1[{p1}]={nums1_demo[p1]} → pos {p}")
        nums1_demo[p] = nums1_demo[p1]; p1 -= 1
    else:
        print(f"    place nums2[{p2}]={nums2_demo[p2]} → pos {p}")
        nums1_demo[p] = nums2_demo[p2]; p2 -= 1
    p -= 1; step_count += 1

print()
print(f"  Merged result     : {result}")
expected_merge = sorted(shift_a + shift_b)
print(f"  Expected (sorted) : {expected_merge}")
print(f"  Correct           : {'✅ YES' if result == expected_merge else '❌ NO'}")


## Step 4 — Timeline visualisation and gap analysis

In [ ]:
# ASCII timeline of merged intervals across the shift
TIMELINE_WIDTH = 60
def timeline_bar(intervals, total, width):
    bar = ["."] * width
    for s, e in intervals:
        lo = int(s / total * width)
        hi = int(e / total * width)
        for i in range(lo, min(hi, width)):
            bar[i] = "█"
    return "".join(bar)

print(f"Maintenance coverage timeline ({SHIFT_LENGTH} min shift):")
print(f"  0 min" + " " * (TIMELINE_WIDTH - 9) + f"{SHIFT_LENGTH} min")
print(f"  |" + timeline_bar(merged, SHIFT_LENGTH, TIMELINE_WIDTH) + "|")
print(f"  Legend: █ = maintenance  . = production")
print()

# Gap analysis
print(f"Gap analysis:")
print(f"  {'Gap':<20}  {'Duration':>10}  {'Starts at':>10}")
print(f"  {'─'*20}  {'─'*10}  {'─'*10}")
all_gaps = []
if merged[0][0] > 0:
    all_gaps.append((0, merged[0][0]))
for i in range(len(merged)-1):
    all_gaps.append((merged[i][1], merged[i+1][0]))
if merged[-1][1] < SHIFT_LENGTH:
    all_gaps.append((merged[-1][1], SHIFT_LENGTH))

for gs, ge in all_gaps:
    dur = ge - gs
    print(f"  [{gs:4d} – {ge:4d}]           {dur:10d} min  at {gs:6d} min")

free_time = sum(e-s for s,e in all_gaps)
print()
print(f"  Total production time: {free_time} / {SHIFT_LENGTH} min  ({free_time/SHIFT_LENGTH*100:.1f}%)")
print(f"  Total maintenance   : {total_covered} / {SHIFT_LENGTH} min  ({total_covered/SHIFT_LENGTH*100:.1f}%)")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " PRODUCTION LINE INTERVAL SCHEDULER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  MERGE SORT" + " "*(W-12) + "║")
print(f"║  {'Intervals sorted':<28}: {N_INTERVALS:<26}║")
print(f"║  {'Merge steps':<28}: {len(merge_steps):<26}║")
print("╠" + "═"*W + "╣")
print("║  INTERVAL MERGING" + " "*(W-18) + "║")
print(f"║  {'Before merge':<28}: {N_INTERVALS} intervals{'':<16}║")
print(f"║  {'After merge':<28}: {len(merged)} windows{'':<18}║")
print(f"║  {'Total maintenance time':<28}: {total_covered} / {SHIFT_LENGTH} min{'':<13}║")
print(f"║  {'Production time':<28}: {free_time} / {SHIFT_LENGTH} min{'':<13}║")
print(f"║  {'Largest gap':<28}: {largest_gap} min{'':<22}║")
print("╠" + "═"*W + "╣")
print("║  IN-PLACE MERGE" + " "*(W-16) + "║")
print(f"║  {'Shift A events':<28}: {N_SHIFT_A:<26}║")
print(f"║  {'Shift B events':<28}: {N_SHIFT_B:<26}║")
merge_ok = (result == expected_merge)
print(f"║  {'Merge correct':<28}: {'YES ✅' if merge_ok else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about sorting in this context?

*Your answer here:*

---

**Q2 — Which sorting concept did you find most important, and why?**
Pick one algorithm or pattern (merge sort, quickselect, interval merging, counting sort…) and explain what problem it solves best.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
